In [1]:
from dotenv import load_dotenv,find_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
import os
load_dotenv()
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
if not GEMINI_API_KEY:
   raise ValueError('GEMINI_API_KEY is not set in the env file')
print('GEMINI_API_KEY loaded successfully')    
llm = ChatGoogleGenerativeAI(api_key=GEMINI_API_KEY,model='gemini-1.5-flash')

GEMINI_API_KEY loaded successfully


In [8]:
from typing_extensions import TypedDict
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, START, END
from IPython.display import Markdown

# Define the graph state
class GraphState(TypedDict):
    story_theme: str
    generated_story: str

def generate_story(state: GraphState) -> GraphState:
    story_theme = state['story_theme']
    
    generate_story_prompt = f'''
    Imagine you are a skilled storyteller tasked with creating a simple and engaging story based on the theme below. 
    
    **Theme:** {story_theme}
    
    Your goal is to generate a detailed and realistic story for 10-year-old children. The story should have:
    - A clear beginning, middle, and end.
    - Easy-to-understand language.
    - Meaningful and detailed sentences.
    - Interesting characters and events.
    
    **Story:**'''
    
    sys_msg = SystemMessage(content="You are a storyteller who writes clear and engaging stories in simple English.")
    hum_msg = HumanMessage(content=generate_story_prompt)
    
    # Ensure `llm` is properly defined before calling invoke
    try:
        resp = llm.invoke([sys_msg, hum_msg])
        state['generated_story'] = resp.content
    except Exception as e:
        state['generated_story'] = f"Error generating story: {str(e)}"
    
    return state

# Create workflow
template = StateGraph(GraphState)
template.add_node('generate_story', generate_story)
template.add_edge(START, 'generate_story')
template.add_edge('generate_story', END)

app = template.compile()

# Story theme input
story_theme = "Once, a rabbit mocked a slow-moving tortoise. The tortoise challenged him to a race. Confident of his speed, the rabbit dashed ahead and took a nap. Meanwhile, the tortoise kept moving steadily. By the time the rabbit woke up, the tortoise had already crossed the finish line."

# Invoke workflow
resp1 = app.invoke({'story_theme': story_theme})

# Display the generated story
Markdown(resp1.get('generated_story', 'No story generated.'))


Barnaby Bunson was a rabbit with fur like a sunset and a nose that twitched with mischief. He loved to zoom across the meadow, a brown blur against the green grass.  One sunny morning, Barnaby hopped past Sheldon the tortoise, who was slowly, slowly making his way towards a patch of juicy clover.

"Goodness, Sheldon!" Barnaby chuckled, his nose twitching. "You're slower than a snail in molasses!  You'll never get there before sunset!"

Sheldon, whose shell was a beautiful mosaic of greens and browns, didn't get upset. He simply smiled a slow, wise smile. "Perhaps," he said, his voice a low rumble, "you're not as fast as you think, Barnaby."

Barnaby scoffed. "Oh, I am! I'm the fastest rabbit in the whole meadow!  Let's have a race! To that big oak tree at the edge of Farmer McGregor's field!"

Sheldon agreed.  Farmer McGregor, who was watching from his porch, chuckled and said, "That's a fine idea! The first one to the oak tree wins a juicy carrot!"

The race began.  Barnaby shot off like an arrow, leaving Sheldon far behind.  Barnaby felt so confident, he decided to take a little nap under a shady bush halfway to the oak tree.  "I've got plenty of time," he thought, yawning.  The sun was warm, and the grass was soft.  Before he knew it, Barnaby was fast asleep.

Meanwhile, Sheldon plodded on.  He didn't run; he didn't even hurry. He just kept moving, one steady step after another.  He passed buzzing bees, colorful butterflies, and even a grumpy badger who grumbled, "What's the rush, Sheldon?"

Hours passed.  Barnaby finally woke up, startled by the sound of buzzing bees. He looked around.  The sun was starting to set, casting long shadows across the meadow.  He saw the oak tree in the distance… and he saw Sheldon, already munching on a delicious carrot offered by Farmer McGregor.  Sheldon had won!

Barnaby was surprised and a little embarrassed.  He learned a valuable lesson that day:  steadiness and perseverance can beat even the greatest speed.  From then on, Barnaby was much kinder to Sheldon, and he learned to appreciate the tortoise's slow and steady way of life.  He even sometimes joined Sheldon on his slow, clover-hunting expeditions.


In [5]:
from typing_extensions import TypedDict
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, START, END
import json
from typing import Sequence
from langgraph.graph.message import add_messages
from langchain_core.messages import BaseMessage
from typing import Annotated
from IPython.display import display, Markdown
import time
from langchain_google_genai import ChatGoogleGenerativeAI
import os

from dotenv import load_dotenv
load_dotenv()
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
if not GEMINI_API_KEY:
   raise ValueError('GEMINI_API_KEY is not set in the env file')
print('GEMINI_API_KEY loaded successfully')    

llm = ChatGoogleGenerativeAI(api_key=GEMINI_API_KEY,model='gemini-1.5-flash')


class MainCharacters(TypedDict):
    name: str
    appearance: str
    characteristics: str


class SupportingCharacters(TypedDict):
    name: str
    appearance: str
    characteristics: str


class ScenesList(TypedDict):
    id: str
    scene: str
    description: str
    narration: str
    img_prompt: str
    object_description:str
     

class GraphState(TypedDict):
    story_theme: str
    generated_story: str
    scene_list: list[ScenesList]
    supporting_characters: list[SupportingCharacters]
    main_characters: list[MainCharacters]
    

def generate_story_char(state: GraphState) -> GraphState:
    story_theme = state["story_theme"]

    new_prompt = """Based on the given story theme, generate a structured list of  scenes where the total narration duration does not exceed 50 seconds.
    and  create a brief description of the main and supporting character, object, or scene. Include specific details about appearance, characteristics . 
    This description will be used to maintain consistency across multiple scenes.
    Each scene must include an **`Object_Description`** field, which provides a short description of key objects, scenery, or important elements in that scene.  
    and also generate a narration that exactly follows this text, starting with 'Once upon a time...' 

      ### Story Theme:
      {story_theme}

      ### Output Format (JSON):
      {{
        "main_characters": [
              {{
                "name": "",
                "appearance": "",
                "characteristics": ""
              }}
            ],
            "supporting_characters": [
              {{
                "name": "",
                "appearance": "",
                "characteristics": ""
              }}
            ],
          "scenes": [
            {{
              "id": "1",
              "scene": "Engaging Beginning",
              "description": "Begin with a captivating moment to grab children's attention.",
              "narration": ""
              "object_description": ""
            }},
         
          ],
          
        }}

      - **Generate  scenes** to ensure a smooth and structured story.
      - **Don't generate more than 5 scenes.**
      - **Start with an engaging scene** to hook the audience immediately. 
      - **Write concise, engaging, and clear narration** to fit within 50 seconds.
      - **End with a meaningful but natural lesson** without making it feel forced.
      - **Use simple and engaging language** suitable for children.
      - **Ensure smooth transitions** so the story flows naturally.

      Return ONLY valid JSON output without any extra formatting or explanations. Do NOT use markdown formatting (e.g., no triple backticks).
      Strictly output **only JSON** without extra text."""

    sys_msg = SystemMessage(
        content="You are an assistant that extracts structured details from a story."
    )
    hum_msg = HumanMessage(content=new_prompt.format(story_theme=story_theme))

    start_time = time.time()
    res = llm.invoke([sys_msg, hum_msg])
    end_time = time.time()

    print(f"Time taken to generate story characters: {end_time - start_time} seconds")
    print(f"llm result :{type(res)},{res}") 
    raw_content = res.content.strip()
    if raw_content.startswith("```") and raw_content.endswith("```"):
        raw_content = raw_content.strip("`")
        if raw_content.lower().startswith('json'):
            raw_content = raw_content[4:].strip() 
    
    try:
        parsed_resp = json.loads(raw_content)
    except json.JSONDecodeError as e:
        raise ValueError(f"Error parsing JSON: {e}\nRaw content: {raw_content}")

    for scene in parsed_resp["scenes"]:
        prompt_template = f"""Create a detailed, photorealistic image of the following scene:
        {scene["description"]}
        
        **Main Characters**:
        {", ".join([f'{char.get("name",'')} - {char.get("appearance",'')}, {char.get("characteristics",'')}' for char in parsed_resp['main_characters'] if parsed_resp['main_characters']])}
 

        **Supporting Characters**:
        {", ".join([f'{supchar.get("name","")} - {supchar.get("appearance","")} - {supchar.get("characteristics","")}' for supchar in parsed_resp['supporting_characters'] if parsed_resp['supporting_characters']])}
        
        **Objects**:
        {scene["object_description"]}
        **Mood & Lighting**: Cinematic, immersive atmosphere with realistic lighting to match the scene's emotions.

        The illustration should capture the story’s essence and atmosphere."""
        scene["img_prompt"] = prompt_template
 
    state["scene_list"] = parsed_resp["scenes"]
 
    state["supporting_characters"] = parsed_resp.get("supporting_characters", [])
    state["main_characters"] = parsed_resp.get("main_characters", [])
    return state

 
workflow = StateGraph(GraphState)
workflow.add_node("generate_story_char", generate_story_char)
workflow.add_edge(START, "generate_story_char")
workflow.add_edge("generate_story_char", END)
app = workflow.compile()
story_theme = "Once, a rabbit mocked a slow-moving tortoise. The tortoise challenged him to a race. Confident of his speed, the rabbit dashed ahead and took a nap. Meanwhile, the tortoise kept moving steadily. By the time the rabbit woke up, the tortoise had already crossed the finish line."
story_theme = """**Clever Mircho**

Once upon a time, in a small village nestled between rolling green hills, there lived a poor but intelligent man named Mircho. He was known for his quick wit and cleverness, but despite his intelligence, he remained poor. He lived in a modest hut with his aging mother, whom he cared for dearly. Though the villagers often made fun of his ragged clothes and simple lifestyle, they also sought his wisdom whenever they found themselves in trouble.

One day, the king of the land issued a proclamation that he was searching for the wisest man in the kingdom. He had grown tired of his ministers, who, despite their scholarly knowledge, failed to solve real-life problems. The king declared that whoever could prove their cleverness through three impossible tasks would be rewarded with riches beyond imagination.

When Mircho heard about the proclamation, he knew this was his chance to change his fate. With nothing to lose, he made his way to the royal palace. Upon arriving, he saw many noblemen and scholars trying their luck, but one by one, they failed and were sent away in disgrace. Mircho stepped forward with confidence.

The king, intrigued by the simple villager, said, “You do not look like a scholar or a wealthy man. What makes you think you are wise enough to complete my tasks?”

Mircho smiled and replied, “Wisdom does not reside in wealth or fine clothing, Your Majesty. Let me attempt your tasks, and you shall see.”

The king, amused by Mircho’s boldness, gave him his first challenge. “Here is your first task: You must bring me a rope made of ashes.”

The courtiers laughed, knowing that ashes crumbled to dust when touched. But Mircho remained calm. He bowed and said, “I shall return with your rope by tomorrow, Your Majesty.”

That evening, Mircho pondered the task. After a moment of thought, he devised a clever solution. He took a thick rope, soaked it in saltwater, and let it dry completely. Then, he carefully placed it in a fire and let it burn completely into ash. Because of the salt residue, the rope retained its shape even as it turned to ash. The next morning, Mircho carefully lifted the delicate, ash-formed rope and carried it to the palace.

The king and his courtiers gasped in astonishment. “You have completed the first task,” the king admitted. “But let’s see how you fare with the next.”

For the second challenge, the king said, “You must fit this entire herd of elephants into a single small clay pot without harming them.”

The onlookers murmured in disbelief. It was an impossible task! But Mircho simply nodded and asked, “Would Your Majesty be so kind as to first remove the elephants from the clay pot they were previously stored in?”

The king frowned. “What nonsense is this? How could the elephants have ever been inside a pot?”

Mircho bowed and said, “If they were never inside a pot, how can I be expected to fit them in?”

The entire court erupted in laughter, and the king himself chuckled, recognizing Mircho’s wit. “Well played,” the king said. “But the third task will not be so easy.”

For the final challenge, the king said, “You must make my palace walk to me.”

The crowd fell silent. Surely this was an impossible task! How could a giant palace be made to move? But Mircho remained unfazed. He bowed and said, “It shall be done, Your Majesty. Please take a seat outside and wait.”

The king, intrigued, did as Mircho asked. Mircho then went to the marketplace and began to announce, “The king has ordered everyone to move the palace to him! Gather your tools and prepare!”

The people, hearing this, were puzzled but followed him to the palace. There, Mircho instructed them to begin dismantling the palace, brick by brick. As soon as the workers started removing the first few bricks, the king shouted, “Stop! What are you doing?”

Mircho smiled and said, “Your Majesty, the palace is beginning to move toward you. I am simply fulfilling your request.”

The king burst into laughter and clapped his hands in admiration. “You have passed all three tests, Mircho. You are truly the cleverest man in the kingdom.”

True to his word, the king rewarded Mircho with gold, land, and a high position in the court. But despite his newfound wealth, Mircho remained humble and wise, always ready to use his intelligence for the good of the people.

And so, Mircho’s cleverness not only changed his fate but also brought prosperity to his village, where he was forever remembered as Clever Mircho. create 8 scence"""
# Invoke workflow
story_themes = "A poor boy"
resp1 = app.invoke({"story_theme": story_themes})
# from IPython.display import display, Markdown
# print(resp1['main_characters'])
 
# print(resp1['supporting_characters'])  
# for scene in resp1['scene_list']:
#     print(scene['id'])
#     print(scene['narration'])
#     # print(scene['Img_prompt'])
print(resp1)    

GEMINI_API_KEY loaded successfully
Time taken to generate story characters: 5.033393621444702 seconds
llm result :<class 'langchain_core.messages.ai.AIMessage'>,content='{\n  "main_characters": [\n    {\n      "name": "Tom",\n      "appearance": "A small boy with messy brown hair and bright, hopeful eyes. He wears patched-up clothes.",\n      "characteristics": "Kind, resourceful, and determined despite his poverty."\n    }\n  ],\n  "supporting_characters": [\n    {\n      "name": "Old Man Fitzwilliam",\n      "appearance": "A kind old man with a long white beard and twinkling eyes. He wears simple but clean clothes.",\n      "characteristics": "Wise, generous, and believes in second chances."\n    }\n  ],\n  "scenes": [\n    {\n      "id": "1",\n      "scene": "Engaging Beginning",\n      "description": "Tom finds a lost puppy.",\n      "narration": "Once upon a time, there lived a poor boy named Tom. One day, while searching for scraps, Tom found a lost, shivering puppy.",\n      "ob

In [10]:


from IPython.display import display, Markdown
# print(resp1['MainCharacters'])
# print(resp1['Objects'])
# print(resp1['SupportingCharacters'])    
for scene in resp1['scene_list']:
    print(scene['id'])
    print(scene['narration'])
    # print(scene['img_prompt'])
    
    

1
Once upon a time, in a small village, lived a poor but clever man named Mircho. He lived with his mother in a small hut.
2
One day, the king announced a contest! He needed the wisest man to solve three impossible tasks.
3
The first task: a rope made of ashes! Mircho soaked a rope in salt water, burned it, and the salty ash held its shape!
4
Next, fit elephants in a small pot! Mircho cleverly asked, 'Where are the elephants?'  The king realized it was a trick!
5
The last task: make the palace walk! Mircho told everyone to move the palace brick by brick. The king laughed, realizing the palace was moving!


In [3]:
import edge_tts

test = """ Once upon a time, in a small village, lived a poor but clever man named Mircho. He lived with his mother in a small hut."""
async def generate_speech(test):
    tts = edge_tts.Communicate(
        test, voice="en-US-JennyNeural", volume="+100%", pitch="+5Hz"
    )
    await tts.save('test1.mp3')

# Directly await in the notebook
await generate_speech(test)


In [19]:
import os
import asyncio
import edge_tts
from pydub import AudioSegment

test = """Once upon a time, in a small village, lived a poor but clever man named Mircho. He lived with his mother in a small hut."""

async def save_voice(text: str, save_path: str):
    tts = edge_tts.Communicate(
        text, voice="en-US-JennyNeural", volume="+100%", pitch="+0Hz"
    )
    await tts.save(save_path)

async def generate_voice_with_bg(text: str):
    try:
        voice_path = "final_voicesss.mp3"
        bg_music_path = "bg_music.mp3"

        # Step 1: Generate voice from text
        await save_voice(text, voice_path)

        # Step 2: Load both audio files
        narration = AudioSegment.from_file(voice_path)
        bg_music = AudioSegment.from_file(bg_music_path)

        # Step 3: Adjust background music volume
        bg_music = bg_music - 40  # Lower dB for background

        # Step 4: Loop and match duration
        while len(bg_music) < len(narration):
            bg_music += bg_music
        bg_music = bg_music[:len(narration)]

        # Step 5: Overlay and export
        final_audio = narration.overlay(bg_music)
        final_audio.export(voice_path, format="mp3")

        print("✅ Final audio saved as 'final_voice.mp3'")

    except Exception as e:
        print(f"❌ Error: {e}")

# Run it
await generate_voice_with_bg(test)


✅ Final audio saved as 'final_voice.mp3'


In [1]:
import numpy as np
import cv2
from PIL import Image, ImageDraw, ImageFont


def split_text_into_segments(text, font, max_width):
    """Split the text into segments that fit within max_width."""
    words = text.split()
    segments = []
    current_segment = ""
    for word in words:
        test_line = current_segment + (" " if current_segment else "") + word
        # Get width of test_line
        w = font.getbbox(test_line)[2]
        if w <= max_width:
            current_segment = test_line
        else:
            if current_segment:
                segments.append(current_segment)
            current_segment = word
    if current_segment:
        segments.append(current_segment)
    return segments


def create_video_with_typewriter_effect(
    scene_data, output_filename, fontsize=60, video_fps=30
):
    # Load the font
    font = ImageFont.truetype("arial.ttf", fontsize)

    # Set video size for YouTube Shorts (1080x1920)
    video_width = 1080
    video_height = 1920
    video_size = (video_width, video_height)

    # Create video writer
    video = cv2.VideoWriter(
        output_filename, cv2.VideoWriter_fourcc(*"mp4v"), video_fps, video_size
    )

    for scene in scene_data:
        narration = scene["narration"]
        scene_duration = scene["audio_duration"]  # in seconds

        # Split narration into segments that fit on one line (based on available width)
        segments = split_text_into_segments(
            narration, font, video_width - 40
        )  # 40 pixels margin
        num_segments = len(segments)
        if num_segments == 0:
            continue

        # Allocate equal duration for each segment
        segment_duration = scene_duration / num_segments
        segment_frames = int(segment_duration * video_fps)

        for segment in segments:
            total_chars = len(segment)
            for frame_idx in range(segment_frames):
                frame = np.zeros((video_height, video_width, 3), dtype=np.uint8)
                frame_pil = Image.fromarray(frame)
                draw = ImageDraw.Draw(frame_pil)

                # Calculate how many characters to show (typewriter effect)
                num_chars = int(frame_idx * total_chars / segment_frames)
                text_to_display = segment[:num_chars]

                # Measure text size and center it
                text_width, text_height = font.getbbox(text_to_display)[2:4]
                x = (video_width - text_width) // 2
                y = (video_height - text_height) // 2

                draw.text((x, y), text_to_display, font=font, fill=(255, 255, 255))
                video.write(np.array(frame_pil))

    video.release()
    print(f"Video saved as {output_filename}")


# Example usage with scene data
scene_data = [
    {
        "narration": "Once upon a time, in a small village, lived a poor but clever man named Mircho. He lived with his mother in a small hut.",
        "audio_duration": 8,
    },
    {
        "narration": "Once upon a time, in a small village, lived a poor but clever man named Mircho. He lived with his mother in a small hut.",
        "audio_duration": 8,
    },
]
output_filename = "typewriter_animation2002.mp4"
create_video_with_typewriter_effect(scene_data, output_filename)

Video saved as typewriter_animation2002.mp4


In [ ]:
from google import genai
from google.genai import types
from PIL import Image
from io import BytesIO
import base64

client = genai.Client(api_key="")

contents = ('Hi, can you create a 3d rendered image of a pig '
            'with wings and a top hat flying over a happy '
            'futuristic scifi city with lots of greenery?')

response = client.models.generate_content(
    model="gemini-2.0-flash-exp-image-generation",
    contents=contents,
    config=types.GenerateContentConfig(
      response_modalities=['Text', 'Image']
    )
)

for part in response.candidates[0].content.parts:
    if part.text:
        print(part.text)  # Print any text response
    elif part.inline_data:
        try:
            # Decode Base64 image data
            image_data = base64.b64decode(part.inline_data.data)
            image = Image.open(BytesIO(image_data))

            # Save and show the image
            image.save("gemini-generated-image.png")
            image.show()
        except Exception as e:
            print(f"Error decoding image: {e}")